# A2.1 – Filtros Espaciales y en Frecuencia

**Nombre del participante:** _(sustituir con tu nombre)_  

**Curso:** Procesamiento de Imágenes y Visión Computacional

**Tema:** 2 – Operaciones sobre Imágenes Digitales

**Actividad:** A2.1 Practicando Operaciones

---

## Paso 1 – Preparación del entorno

Ejecuta la siguiente celda para instalar las dependencias necesarias.

In [ ]:
# Instalación de dependencias (descomentar si es necesario)
# !pip install numpy matplotlib opencv-python scipy

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("Librerías cargadas correctamente.")
print(f"  OpenCV  : {cv2.__version__}")
print(f"  NumPy   : {np.__version__}")

---
## Paso 2 – Carga y visualización de la imagen

Sube el archivo `imagen_tema2.jpg` a tu entorno (en Colab: ícono de carpeta → subir archivo)  
o colócalo en la misma carpeta que este notebook si trabajas en local.

In [ ]:
# ── Cargar imagen ──────────────────────────────────────────────────────────────
IMAGE_PATH = 'imagen_tema2.jpg'   # <-- cambia la ruta si es necesario

img_bgr = cv2.imread(IMAGE_PATH)

if img_bgr is None:
    raise ValueError(f"No se pudo cargar '{IMAGE_PATH}'. Verifica nombre y ruta.")

img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

print("Dimensiones (alto × ancho):", img_gray.shape)
print("Tipo de dato              :", img_gray.dtype)

# ── Visualización ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].imshow(img_rgb)
axes[0].set_title("Imagen en color (RGB)", fontsize=13)
axes[0].axis('off')

axes[1].imshow(img_gray, cmap='gray')
axes[1].set_title("Escala de grises", fontsize=13)
axes[1].axis('off')

plt.tight_layout()
plt.show()

### 📝 Registro de observaciones – Paso 2

| Campo | Valor |
|---|---|
| Dimensiones (alto × ancho) | _(completar)_ |
| Tipo de dato | _(completar)_ |
| Descripción de la escena | _(completar)_ |

---
## Paso 3 – Filtros espaciales: suavizado y detección de bordes

In [ ]:
# ── Kernels y filtros ─────────────────────────────────────────────────────────

# Suavizado promedio 3×3
kernel_promedio = np.ones((3, 3), np.float32) / 9.0
img_promedio = cv2.filter2D(img_gray, -1, kernel_promedio)

# Suavizado gaussiano 5×5  (sigma=1)
img_gauss = cv2.GaussianBlur(img_gray, (5, 5), 1.0)

# Bordes Sobel (combinado X + Y)
sobelx   = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
sobely   = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
img_sobel = cv2.magnitude(sobelx, sobely)

# Laplaciano
img_lap = cv2.Laplacian(img_gray, cv2.CV_64F, ksize=3)

# ── Visualización ─────────────────────────────────────────────────────────────
titles  = ["Original (grises)", "Suavizado promedio",
           "Suavizado gaussiano", "Bordes Sobel", "Laplaciano"]
images  = [img_gray, img_promedio, img_gauss, img_sobel, img_lap]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for i, (img, title) in enumerate(zip(images, titles)):
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(title, fontsize=12)
    axes[i].axis('off')

axes[-1].axis('off')   # celda vacía en la cuadrícula
plt.tight_layout()
plt.show()

### 📝 Observaciones – Filtros espaciales

| Filtro | Kernel / Operación | Efecto observado |
|---|---|---|
| Promedio 3×3 | `np.ones((3,3))/9` | _(completar)_ |
| Gaussiano 5×5 | `GaussianBlur(σ=1)` | _(completar)_ |
| Sobel | Derivada 1ª orden X+Y | _(completar)_ |
| Laplaciano | Derivada 2ª orden | _(completar)_ |

---
## Paso 4 – Análisis comparativo de filtros espaciales

Compara histogramas de píxeles para cuantificar el efecto de cada filtro.

In [ ]:
# ── Histogramas de intensidad ─────────────────────────────────────────────────
filtros_a_comparar = [
    (img_gray,     "Original",          "#333333"),
    (img_promedio, "Prom. 3×3",          "#2196F3"),
    (img_gauss,    "Gaussiano 5×5",      "#4CAF50"),
]

plt.figure(figsize=(10, 4))
for img, label, color in filtros_a_comparar:
    hist = cv2.calcHist([img.astype(np.uint8)], [0], None, [256], [0, 256])
    plt.plot(hist.ravel(), label=label, color=color, alpha=0.8)

plt.title("Histogramas – filtros de suavizado")
plt.xlabel("Intensidad")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.show()

### 📝 Análisis – Paso 4

_(Describe qué cambia en los histogramas y qué efecto visual produce cada filtro sobre la imagen. 3–5 líneas.)_

---
## Paso 5 – Dominio de la frecuencia: TDF 2D

In [ ]:
# ── Transformada de Fourier 2D ─────────────────────────────────────────────────
f      = np.fft.fft2(img_gray)
fshift = np.fft.fftshift(f)

magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title("Imagen original (grises)", fontsize=13)
axes[0].axis('off')

axes[1].imshow(magnitude_spectrum, cmap='inferno')
axes[1].set_title("Espectro de magnitud (log)", fontsize=13)
axes[1].axis('off')

plt.tight_layout()
plt.show()

---
## Paso 6 – Filtros pasa-bajas y pasa-altas en frecuencia

In [ ]:
# ── Máscaras circulares ────────────────────────────────────────────────────────
rows, cols = img_gray.shape
crow, ccol = rows // 2, cols // 2
radio      = min(rows, cols) // 8   # ajusta este valor para variar el corte

mask_low  = np.zeros((rows, cols), np.uint8)
cv2.circle(mask_low, (ccol, crow), radio, 1, -1)
mask_high = 1 - mask_low

# ── Filtrar y reconstruir ──────────────────────────────────────────────────────
f_low  = fshift * mask_low
f_high = fshift * mask_high

img_low  = np.abs(np.fft.ifft2(np.fft.ifftshift(f_low)))
img_high = np.abs(np.fft.ifft2(np.fft.ifftshift(f_high)))

# ── Visualización ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

datos = [
    (img_gray, "Original (grises)"),
    (img_low,  f"Pasa-bajas (r={radio}px)"),
    (img_high, f"Pasa-altas (r={radio}px)"),
]

for ax, (img, title) in zip(axes, datos):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Visualizar también las máscaras
fig2, axes2 = plt.subplots(1, 2, figsize=(8, 3))
axes2[0].imshow(mask_low,  cmap='gray'); axes2[0].set_title("Máscara pasa-bajas");  axes2[0].axis('off')
axes2[1].imshow(mask_high, cmap='gray'); axes2[1].set_title("Máscara pasa-altas"); axes2[1].axis('off')
plt.tight_layout()
plt.show()

### 📝 Observaciones – Filtros en frecuencia

| Filtro | Efecto en la imagen |
|---|---|
| Espectro de magnitud | _(completar)_ |
| Pasa-bajas | _(completar)_ |
| Pasa-altas | _(completar)_ |

---
## Paso 7 – Interpretación y comparación de resultados

In [ ]:
# ── Tabla comparativa visual ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

pares = [
    # fila 0: dominio espacial
    (img_gray,     "Original"),
    (img_promedio, "Espacial: Prom. 3×3"),
    (img_gauss,    "Espacial: Gaussiano"),
    (img_sobel,    "Espacial: Sobel"),
    # fila 1: dominio frecuencial
    (magnitude_spectrum, "Espectro magnitud"),
    (img_low,            "Freq: Pasa-bajas"),
    (img_high,           "Freq: Pasa-altas"),
    (img_lap,            "Espacial: Laplaciano"),
]

for ax, (img, title) in zip(axes.ravel(), pares):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.suptitle("Comparativa completa – filtros espaciales vs. frecuenciales",
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 📝 Reflexión comparativa – Paso 7

_(Redacta entre 10 y 15 líneas conectando los siguientes puntos:)_

1. **Suavizado promedio / gaussiano** → información atenuada vs. resaltada.
2. **Bordes (Sobel, Laplaciano)** → información atenuada vs. resaltada.
3. **Pasa-bajas en frecuencia** → equivalencia con suavizado espacial.
4. **Pasa-altas en frecuencia** → equivalencia con detección de bordes.
5. Relación con la propiedad de sistemas LTI: convolución en espacio ↔ producto en frecuencia.

---

_(Escribe tu reflexión aquí.)_

---
## Extensión de la actividad – Elige una variante

Descomenta y ejecuta **solo la variante que elijas**.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VARIANTE A – Kernels propios
# ═══════════════════════════════════════════════════════════════════════════════

# Kernel de suavizado propio (box blur ponderado)
kernel_suave_propio = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], np.float32) / 16.0

# Kernel de bordes horizontales propio
kernel_bordes_h = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], np.float32)

img_suave_propio  = cv2.filter2D(img_gray, -1, kernel_suave_propio)
img_bordes_h      = cv2.filter2D(img_gray, cv2.CV_64F, kernel_bordes_h)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
pares_a = [
    (img_gray,         "Original"),
    (img_suave_propio, "Suavizado propio"),
    (img_bordes_h,     "Bordes H propios"),
    (img_sobel,        "Sobel (referencia)"),
]
for ax, (img, title) in zip(axes, pares_a):
    ax.imshow(img, cmap='gray'); ax.set_title(title, fontsize=11); ax.axis('off')
plt.suptitle("Variante A – Kernels propios vs. Sobel", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Kernel suavizado propio:\n", kernel_suave_propio)
print("Kernel bordes horizontales:\n", kernel_bordes_h)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VARIANTE B – Filtro de mediana (ruido sal y pimienta)
# ═══════════════════════════════════════════════════════════════════════════════

def add_salt_pepper(image, amount=0.03):
    """Agrega ruido sal y pimienta a una imagen en grises."""
    noisy = image.copy()
    n_pixels = image.size
    # Sal (blanco)
    coords = [np.random.randint(0, i, int(n_pixels * amount)) for i in image.shape]
    noisy[tuple(coords)] = 255
    # Pimienta (negro)
    coords = [np.random.randint(0, i, int(n_pixels * amount)) for i in image.shape]
    noisy[tuple(coords)] = 0
    return noisy

img_noisy   = add_salt_pepper(img_gray, amount=0.04)
img_median3 = cv2.medianBlur(img_noisy, 3)
img_median5 = cv2.medianBlur(img_noisy, 5)
img_gauss_n = cv2.GaussianBlur(img_noisy, (5, 5), 1.0)  # comparación

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
pares_b = [
    (img_noisy,    "Con ruido S&P"),
    (img_median3,  "Mediana 3×3"),
    (img_median5,  "Mediana 5×5"),
    (img_gauss_n,  "Gaussiano 5×5 (ref.)"),
]
for ax, (img, title) in zip(axes, pares_b):
    ax.imshow(img, cmap='gray'); ax.set_title(title, fontsize=11); ax.axis('off')
plt.suptitle("Variante B – Filtro de mediana vs. ruido sal y pimienta", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 📝 Comentario sobre la extensión

_(Qué aprendiste con la variante elegida. 5–8 líneas.)_

---
## Paso 8 – Reflexión final _(mínimo ½ cuartilla)_

Aborda los siguientes puntos:

**a)** Importancia de entender los sistemas LTI, la convolución y la TDF para la visión por computadora.

**b)** Relación entre los filtros clásicos (Sobel, gaussiano, etc.) y los filtros aprendidos en las primeras capas de una CNN.

---

_(Escribe tu reflexión aquí.)_

---
*Fin del notebook – T2.A2.1*